# Exercise 1c — Text Generation
**Paper:** Vaswani et al., *"Attention Is All You Need"* (2017)  
**Focus Section:** Positional Encoding  
**Model:** Qwen/Qwen3-0.6B (https://huggingface.co/Qwen/Qwen3-0.6B)

---

## Overview

This notebook explores how a pre-trained tranformer model can assist in expanding and simplifying content from an academic research paper.

---

## Setup

In [ ]:
import torch
import textwrap
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


## 2. Prompt Design

The prompt is constructed as an **instruction-style input**, which matches how Qwen3-0.6B was trained.

**Selected sentence from the paper:**
> *"We chose the sinusoidal version because it may allow the model to extrapolate to sequence lengths longer than the ones encountered during training."*

The prompt wraps this sentence in an explicit instruction to generate a beginner-friendly explanation.

In [ ]:
prompt = """Explain the following concept from the Attention Is All You Need paper
            in simple terms for a beginner: We chose the sinusoidal version because it may allow the model to extrapolate
            to sequence lengths longer than the ones encountered during training"""

messages = [
    {"role": "user", "content": prompt}
]

## 3. Model Loading

**Model:** `Qwen/Qwen3-0.6B`  
**Architecture:** Language Model (decoder-only), 0.6B parameters  


The tokenizer and model are loaded separately, `AutoModelForCausalLM` gives direct control over generation parameters, which is essential for this experiment.

In [ ]:
model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto", trust_remote_code=True).to(device)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## 4. Baseline Generation

A single baseline generation is performed to verify the prompt produces meaningful output and to establish a reference for comparison.

The chat template is applied with `enable_thinking=False` to use non-thinking mode.

In [ ]:
# Apply chat template and tokenize
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate with baseline parameters
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=300
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
wrapped_content = textwrap.fill(content, width=80)
print("Baseline output:\n", wrapped_content)


Baseline output:
 Sure! Here's a simple explanation:  In the paper *"Attention Is All You Need"*
by Vaswani et al., they mention that they chose a **sinusoidal** version of the
attention mechanism instead of just a regular sine wave. This is because the
model might be better at **extending the sequence** (the time steps in the
input) when the sequence length is longer than what they trained on.  So, if the
training data only has a few time steps, the sinusoidal version could help the
model **learn to predict the next few steps** even if the actual training
sequence is longer. This makes it more flexible and powerful for tasks like
language modeling.


## 5. Parameter Experiments

### Parameter Overview

| Parameter | Role | Effect of increasing |
|---|---|---|
| `temperature` | Scales logit distribution before sampling | Higher = more random/creative, lower = more predictable |
| `top_k` | Limits vocabulary to k most probable tokens | Higher = more diverse word choice, lower = more conservative |
| `top_p` | Nucleus sampling, cumulative probability threshold | Higher = wider token pool, lower = more focused |
| `repetition_penalty` | Penalizes previously seen tokens | Higher = avoids repetition, too high = incoherent output |
| `max_new_tokens` | Maximum output length | Controls verbosity |

### Experimental Design

Each experiment changes **one parameter at a time** from the baseline, keeping all others constant.

**Baseline (recommended by Qwen3 model card):**
- `temperature=0.7`, `top_k=20`, `top_p=0.8`, `repetition_penalty=1.0`

`do_sample=True` is required for all experiments, without it, the model uses greedy decoding and temperature/top_k/top_p have no effect.

In [ ]:
experiments = [
    {"name": "baseline",     "temperature": 0.7, "top_k": 20,  "top_p": 0.8,  "repetition_penalty": 1.0, "max_new_tokens": 200},
    {"name": "high_temp",    "temperature": 0.9, "top_k": 20,  "top_p": 0.8,  "repetition_penalty": 1.0, "max_new_tokens": 200},
    {"name": "low_temp",     "temperature": 0.3, "top_k": 20,  "top_p": 0.8,  "repetition_penalty": 1.0, "max_new_tokens": 200},
    {"name": "high_top_k",   "temperature": 0.7, "top_k": 90, "top_p": 0.8,  "repetition_penalty": 1.0, "max_new_tokens": 200},
    {"name": "low_top_k",    "temperature": 0.7, "top_k": 5, "top_p": 0.8,  "repetition_penalty": 1.0, "max_new_tokens": 200},
    {"name": "rep_penalty",  "temperature": 0.7, "top_k": 20,  "top_p": 0.8,  "repetition_penalty": 2.0, "max_new_tokens": 200},
    {"name": "high_top_p", "temperature": 0.7, "top_k": 20, "top_p": 0.95, "repetition_penalty": 1.0, "max_new_tokens": 200},
    {"name": "low_top_p", "temperature": 0.7, "top_k": 20, "top_p": 0.4, "repetition_penalty": 1.0, "max_new_tokens": 200},
]

In [ ]:
results = {}

for exp in experiments:
    print(f"Running experiment: {exp['name']}")

    generated_ids = model.generate(
        **model_inputs,
        do_sample=True,
        max_new_tokens=exp['max_new_tokens'],
        temperature=exp['temperature'],
        top_k=exp['top_k'],
        top_p=exp['top_p'],
        repetition_penalty=exp['repetition_penalty'],
    )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

    results[exp['name']] = content
    print(f"Done: {exp['name']}\n")

Running experiment: baseline
Done: baseline

Running experiment: high_temp
Done: high_temp

Running experiment: low_temp
Done: low_temp

Running experiment: high_top_k
Done: high_top_k

Running experiment: low_top_k
Done: low_top_k

Running experiment: rep_penalty
Done: rep_penalty

Running experiment: high_top_p
Done: high_top_p

Running experiment: low_top_p
Done: low_top_p



## 6. Results & Analysis

### Comparison with the Original Paper

Most outputs correctly captured the main idea. However, the level of accuracy varied:

- **baseline & low_temp & low_top_p** stayed closest to the paper
- **high_temp** simplified too much and lost the positional encoding context
- **high_top_k** introduced a wrong comparison to ReLU
- **rep_penalty** broke down completely

---

### How Parameters Affected the Output

**Temperature** (tested: 0.3 / 0.7 / 0.9)
Lower values produced safer, more repetitive text. Higher values gave more
creative language but drifted away from the paper's meaning.

**top_k** (tested: 5 / 20 / 90)
Results varied across runs due to the randomness of sampling. In one run
high_top_k produced the best output, in another it hallucinated a ReLU
comparison. This shows that extreme top_k values make outputs less
reproducible and reliable.

**top_p** (tested: 0.4 / 0.8 / 0.95)
Both high and low top_p produced similar, coherent outputs with no meaningful
difference in quality. For short academic explanations like this, top_p has
limited impact compared to temperature or top_k.

**repetition_penalty** (tested: 1.0 / 2.0)
Setting it to 2.0 produced the most unstable output. While the text remained
partially on topic, the writing became fragmented. The model struggled to reuse key technical terms naturally, leading to broken sentence structures.

---

### Limitations

- The model can introduce wrong information even at moderate
  parameter settings
- The model tends to oversimplify and lose technical precision
- Baseline settings worked best overall for this task
- Generated text should always be reviewed

In [ ]:
for name, content in results.items():
    print(f"--- Experiment: {name} ---")
    wrapped_content = textwrap.fill(content, width=80)
    print(wrapped_content)
    print("\n" + "="*50 + "\n")

--- Experiment: baseline ---
Sure! Here's a simple explanation of what the paper says in your terms:  We
chose the **sinusoidal version** because it might help the model learn to
**extend** or go beyond the sequence lengths it was trained on.   In other
words, if the model was trained on a small number of sentences, the sinusoidal
version might help it generalize to longer sequences in the future.


--- Experiment: high_temp ---
Sure! Here's a simple explanation for a beginner:  **In the paper "Attention Is
All You Need," the author says that they chose the **sine wave version** because
it might help the model learn more patterns in the data.**  Imagine you're
learning to read a song. If you only listen to the song, you might not know
where the next note goes. But if you hear a sine wave (which is a smooth,
repeating sound), you can predict where the next note might come next, even if
the song is longer. This is what the model does when it uses a sine wave as the
input. It helps the mo